# Figuras do modelo de vacinação

Este notebook **não roda simulações**. Ele lê os arquivos `results/<código>.npz` (um por configuração, gerados por `analyses/juntar_pedacos.py`) e desenha as figuras 05 a 19 chamando as mesmas funções de `analyses/figuras_vacinacao.py` e `analyses/figuras_complementares.py`. Assim o notebook e o script do cluster sempre produzem figuras idênticas.

**Configurações do ensemble** (52 arquivos):

- *principal* (30): os 15 cenários `F0`–`F7` e `M0`–`M6` com 9 infectados iniciais em bloco central (sufixo `C`, por exemplo `F1C`) e espalhados pela grade (sufixo `E`, por exemplo `F1E`);
- *semente* (10): `F0` e `M0` com 1, 25 e 81 infectados iniciais (`F0C_I25`, `F0E_I25`, ...);
- *conexões* (12): `F0` e `M0` com $K = 0$, 1 e 10 encontros de mercado por passo (`F0C_K0`, `M0E_K10`, ...).

**Antes de executar**

1. Traga os arquivos `.npz` já reunidos do cluster para a pasta `results/` do repositório local (comando `rsync` no final deste notebook). Os pedaços de `results/pedacos/` não são necessários.
2. O ambiente precisa do Jupyter além de NumPy, SciPy e Matplotlib, por exemplo `conda install -n small-world -c conda-forge jupyterlab`.
3. Execute as células de cima para baixo. A célula de carregamento é a mais lenta, porque lê todos os arquivos e calcula as estatísticas entre seeds.

**Como alterar uma figura**

- *Parâmetros* (célula de configuração): onde gravar, resolução, se grava os PNG, tipo de banda (`dp` ou `ep`), condições iniciais mostradas.
- *Cores e retoques pontuais*: as constantes dos módulos (`FV.COR_GAUSSIANA`, `FC.CORES_ESTADOS` etc.) e a figura devolvida podem ser alteradas antes de `mostrar(...)`; há um exemplo na seção "Retoques sem editar o módulo".
- *Mudança estrutural* (rótulos, painéis, layout): edite a função correspondente em `analyses/figuras_vacinacao.py` (figuras 05 a 12) ou `analyses/figuras_complementares.py` (figuras 13 a 19) e reexecute a célula da figura (a extensão `autoreload` recarrega os módulos). Se mudar `resumir_cenario`, reexecute também a célula de carregamento. Mudanças feitas nos módulos valem também para o script, e por isso ficam reprodutíveis para o grupo depois do commit.

In [ ]:
from pathlib import Path
import sys


def achar_raiz():
    # Sobe a partir da pasta atual até achar a raiz do repositório (onde existe src/model_vacinacao.py).
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "model_vacinacao.py").exists():
            return p
    raise FileNotFoundError("Raiz do repositório não encontrada: abra o notebook de dentro do repositório small-world.")


RAIZ = achar_raiz()

# ----- configuração -----
RESULTADOS = RAIZ / "results"           # pasta com F0C.npz ... M0E_K10.npz
SAIDA = RAIZ / "analyses" / "plots"     # onde gravar os PNG quando SALVAR = True
SALVAR = False                          # False: só mostra na tela, não grava nada
DPI_TELA = 110                          # resolução na tela
DPI_SALVAR = 300                        # resolução dos PNG gravados (o script do cluster usa 300)
BANDAS = ("dp", "ep")                   # "dp" = média ± desvio padrão; "ep" = média ± erro padrão da média
CONDICOES = ("C", "E")                  # "C" = 9 infectados em bloco central; "E" = 9 espalhados
SEED_GRADES = 0                         # execução mostrada na figura 12 (as seeds 0, 1 e 2 guardam as grades)

sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "analyses"))
print("Raiz do repositório:", RAIZ)
print("Resultados em      :", RESULTADOS, "(existe)" if RESULTADOS.exists() else "(NÃO EXISTE)")

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import figuras_vacinacao as FV
import figuras_complementares as FC

# Os módulos forçam o backend Agg (para o cluster); aqui reativamos a exibição dentro do notebook.
%matplotlib inline
plt.rcParams["figure.dpi"] = DPI_TELA

In [ ]:
def mostrar(fig, nome):
    # Mostra a figura no notebook e, se SALVAR = True, grava o PNG em SAIDA.
    if fig is None:
        print(f"[aviso] sem dados para {nome}")
        return
    display(fig)
    if SALVAR:
        SAIDA.mkdir(parents=True, exist_ok=True)
        fig.savefig(SAIDA / nome, dpi=DPI_SALVAR, bbox_inches="tight")
        print("[ok] gravado", SAIDA / nome)
    plt.close(fig)

## Carregar os resultados

`FV.carregar` lê cada `results/<código>.npz` e guarda as estatísticas usadas nas figuras (média, desvio padrão e erro padrão entre seeds) e algumas grandezas por seed. Configurações sem arquivo são avisadas e ficam fora das figuras. `resumos` é indexado pelo código da configuração (`resumos["F1E"]`); `FV.por_condicao(resumos, "C")` devolve os 15 cenários de uma condição inicial, indexados pelo cenário (`"F1"`).

In [ ]:
resumos = FV.carregar(RESULTADOS)
print()
print(len(resumos), "configurações carregadas:", ", ".join(resumos) if resumos else "nenhuma")

## Resumo por configuração

Conferência rápida antes das figuras: número de seeds, primeiras doses aplicadas em média, fração de execuções em que a epidemia se extinguiu antes do horizonte, passo mediano de extinção entre elas, fração de surtos falhos (menos de 10% da população infectada ao menos uma vez) e fração da população infectada ao menos uma vez. Em execuções extintas a curva de infectados fica em zero até o horizonte, e a média entre seeds mistura execuções extintas com execuções endêmicas.

In [ ]:
print(f"{'config.':10}{'seeds':>6}{'doses 1':>9}{'extintas':>10}{'passo mediano':>15}{'surtos falhos':>15}{'já infectados':>15}")
for codigo, r in resumos.items():
    ext = r["t_extincao"][r["t_extincao"] >= 0]
    mediana = str(int(np.median(ext))) if ext.size else "-"
    print(f"{codigo:10}{r['n_seeds']:>6}{float(r['doses1_total']['media']):>9.0f}"
          f"{100 * r['extintas']:>9.0f}%{mediana:>15}{100 * r['falho_seeds'].mean():>14.0f}%"
          f"{100 * r['final_seeds'].mean():>14.1f}%")

# Figuras 05 a 12, por condição inicial

Cada figura desta parte sai uma vez para cada condição inicial de `CONDICOES`; a condição aparece em cinza acima do título.

## Figura 05: Flu-like, infectados ao longo do tempo

Três painéis, um por regime de vacinação (instantânea em $t=0$, instantânea em $t=60$, contínua com 50 doses por passo). Cada painel compara sem vacina, alocação aleatória e alocação gaussiana (grupos de risco). No painel de $t=0$ entra também o controle sem reforço (`F7`). A linha é a média entre seeds e a faixa sombreada é a banda escolhida.

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    for banda in BANDAS:
        mostrar(FV.anotar_condicao(FV.fig_infectados(fam, "flu", banda), cond), f"05_flu_infectados_{banda}_{cond}.png")

## Figura 06: Measles-like, infectados ao longo do tempo

Mesma estrutura da figura 05. Como a epidemia termina cedo no Measles-like, o eixo do tempo é cortado depois do último passo com infectados, e o título avisa o corte.

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    for banda in BANDAS:
        mostrar(FV.anotar_condicao(FV.fig_infectados(fam, "measles", banda), cond), f"06_measles_infectados_{banda}_{cond}.png")

## Figura 07: Flu-like, primeira onda

Aproximação da primeira onda. A linha pontilhada de cada cenário marca o fim da primeira onda, definido sobre a média do ensemble (média móvel de 21 passos, ignorando máximos abaixo de 10% do máximo global).

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    for banda in BANDAS:
        mostrar(FV.anotar_condicao(FV.fig_infectados(fam, "flu", banda, primeira=True), cond), f"07_flu_primeira_onda_{banda}_{cond}.png")

## Figura 08: participação dos vacinados

Porcentagem das infecções sofridas e das transmissões feitas por indivíduos que já tinham recebido ao menos uma dose. Barras azuis: simulação inteira; barras hachuradas: só a primeira onda (Flu-like). Essa porcentagem depende de quantas pessoas estão vacinadas, então não compara riscos; para isso servem as taxas da figura 09.

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    for banda in BANDAS:
        mostrar(FV.anotar_condicao(FV.fig_participacao(fam, banda), cond), f"08_participacao_vacinados_{banda}_{cond}.png")

## Figura 09: taxas por faixa de imunidade

Infecções por pessoa-passo exposta e transmissões por pessoa-passo infectada, separadas pela faixa de imunidade $m$ do indivíduo no instante do evento (nunca vacinado, $m\geq 0{,}90$, $0{,}75\leq m<0{,}90$, $0{,}50\leq m<0{,}75$, $m<0{,}50$). Escala logarítmica. Só entram passos com ao menos um infectado; faixas sem pessoas-passo ou sem eventos não aparecem.

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    for banda in BANDAS:
        mostrar(FV.anotar_condicao(FV.fig_taxas(fam, banda), cond), f"09_taxas_por_faixa_{banda}_{cond}.png")

## Figura 10: faixas dos vacinados e reforços (Flu-like)

Linhas superiores: fração dos vacinados em cada faixa de imunidade ao longo do tempo, para alocação aleatória e gaussiana em cada regime. Linha inferior: doses de reforço aplicadas por passo.

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    for banda in BANDAS:
        mostrar(FV.anotar_condicao(FV.fig_faixas_reforcos(fam, banda), cond), f"10_flu_faixas_e_reforcos_{banda}_{cond}.png")

## Figura 11: infecções por indivíduo (Flu-like)

Distribuição do número de infecções sofridas por indivíduo ao longo do horizonte, média entre seeds (sem banda).

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    mostrar(FV.anotar_condicao(FV.fig_infeccoes_por_individuo(fam), cond), f"11_flu_infeccoes_por_individuo_{cond}.png")

## Figura 12: mapas de imunidade de uma execução

Imunidade $m$ de cada sítio (azul) e sítios infectados (laranja) em uma única execução, em $t = 0$, 60, 300 e no horizonte, para `F1` e `F2`.

In [ ]:
for cond in CONDICOES:
    fam = FV.por_condicao(resumos, cond)
    mostrar(FV.anotar_condicao(FV.fig_grades(fam, seed=SEED_GRADES), cond), f"12_grades_imunidade_{cond}.png")

# Figuras 13 a 19

## Figura 13: evolução espacial em quatro estados

Uma linha por cenário e uma coluna por instante ($t = 0$, 50, 100, 150, 300, 600 e o horizonte). Cada sítio está em um de quatro estados: **suscetível** (não infectado, com imunidade $m < 0{,}5$), **infectado**, **recuperado** ($m \geq 0{,}5$ com imunidade natural maior ou igual à vacinal) e **vacinado** ($m \geq 0{,}5$ com imunidade vacinal maior). Em cada cenário, a execução mostrada é a seed, entre as que guardam grades, cujo tamanho final está mais próximo da mediana do ensemble.

Em $t = 0$ ninguém aparece como vacinado, mesmo nos cenários com vacinação em $t = 0$: a curva de imunidade começa em zero e leva cerca de dois passos para passar de 0,5.

In [ ]:
for doenca in ("flu", "measles"):
    for cond in CONDICOES:
        mostrar(FC.fig_evolucao_espacial(resumos, doenca, cond), f"13_evolucao_espacial_{doenca}_{cond}.png")

## Figura 14: comparação com as EDOs de campo médio

Ensemble sem vacina (linhas cheias, com banda) contra a EDO de campo médio (tracejado), para $S$, $I$ e $R$:

$$\frac{dS}{dt} = -\beta\frac{SI}{N} + \omega R,\qquad \frac{dI}{dt} = \beta\frac{SI}{N} - \gamma I,\qquad \frac{dR}{dt} = \gamma I - \omega R,$$

com $\beta = \alpha\,(8\,W_{\text{local}} + K\,W_{\text{mercado}})\,\langle S_0\rangle\langle T_0\rangle$, $\gamma = p_{\text{rec}}$ e $\omega = 0$ no Measles-like (SIR). No Flu-like (SIRS), $\omega = 1/(\tau_n \ln 2)$ é o inverso do tempo que a imunidade natural leva para cair abaixo de 0,5. Essa troca é uma aproximação: no autômato, o tempo é fixo e a proteção cai aos poucos, e a EDO supõe uma taxa constante de perda. A EDO também ignora a estrutura espacial e a heterogeneidade de $S_0$ e $T_0$.

In [ ]:
for banda in BANDAS:
    mostrar(FC.fig_edo(resumos, banda), f"14_comparacao_edo_{banda}.png")

## Figura 15: semente inicial

Cenários sem vacina com $I_0 = 1$, 9, 25 e 81 infectados iniciais, em bloco central (linha cheia, círculo) ou espalhados (tracejado, quadrado).

- Esquerda: média de infectados ao longo do tempo.
- Centro: fração de surtos falhos (menos de 10% da população infectada), com a previsão de um processo de ramificação em mistura homogênea, $q^{I_0}$. Aqui $q$ é a probabilidade de a linhagem de um único infectado se extinguir, calculada com as regras do modelo (período infeccioso geométrico, infecções de Poisson por passo), com $T_0$ heterogêneo (linha cheia) ou igual à média (pontilhado). A previsão vale para população infinita e sem estrutura espacial.
- Direita: instante do pico máximo de infectados, sem contar os surtos falhos.

In [ ]:
for banda in BANDAS:
    mostrar(FC.fig_semente(resumos, banda), f"15_semente_inicial_{banda}.png")

## Figura 16: conexões globais

Cenários sem vacina com $K = 0$, 1, 3 e 10 encontros de mercado por passo. Com $K = 0$ só há contágio entre vizinhos.

- Esquerda: média de infectados, com o tempo em escala logarítmica (as epidemias com $K$ pequeno são muito mais lentas).
- Centro: fração da população infectada ao menos uma vez até o horizonte.
- Direita: suscetíveis no horizonte, como fração dos sítios ($p$) e como fração dos sítios no maior aglomerado ($P_\infty$); a linha tracejada é o limiar de percolação $p_c \approx 0{,}407$.

In [ ]:
for banda in BANDAS:
    mostrar(FC.fig_conexoes(resumos, banda), f"16_conexoes_globais_{banda}.png")

## Figura 17: vacinação preventiva, reativa e contínua

Um painel reúne os três regimes: preventiva ($t = 0$), reativa ($t = 60$, linha vertical) e contínua (50 doses por passo, de $t = 0$ até cerca de $t = 100$). A cor é o regime; linha cheia é alocação aleatória e tracejada é alocação gaussiana. A primeira versão não tem bandas, para as sete curvas ficarem legíveis; as versões seguintes mostram só a alocação aleatória, com a banda.

A vacinação reativa fica em $t = 60$ porque o que define "reativa" é a fase da epidemia (a subida da primeira onda), e não uma fração do horizonte: o tempo característico da epidemia é fixado por $p_{\text{rec}}$, $\alpha$ e $K$.

In [ ]:
for cond in CONDICOES:
    mostrar(FC.fig_regimes(resumos, cond), f"17_regimes_vacinacao_{cond}.png")
    for banda in BANDAS:
        mostrar(FC.fig_regimes(resumos, cond, banda), f"17_regimes_vacinacao_bandas_{banda}_{cond}.png")

## Figura 18: percolação dos suscetíveis

Os suscetíveis formam aglomerados (sítios ligados por vizinhança de Moore, com contorno periódico). O parâmetro de controle é a fração de sítios suscetíveis, $p$; o parâmetro de ordem é a fração dos sítios no maior aglomerado, $P_\infty$. Na rede quadrada com vizinhança de Moore, o limiar de percolação de sítios é $p_c = 1 - 0{,}592746 \approx 0{,}407$.

- Parâmetro de ordem: cada ponto é uma seed em um dos instantes gravados. A curva cinza é a percolação de sítios ocupados ao acaso na mesma rede $100\times100$; desvios em relação a ela indicam correlações espaciais entre os suscetíveis que sobram. O painel de baixo mostra a fração das medidas em que o maior aglomerado dá a volta no toro, que é o critério de percolação em rede finita.
- Mapas: estados em três instantes e, na última coluna, os aglomerados de suscetíveis, com o maior em azul escuro e o material inerte (recuperados e vacinados) em cinza.

O limiar geométrico vale para contágio só entre vizinhos. Com $K > 0$, os encontros de mercado ligam aglomerados distantes, então um aglomerado de suscetíveis isolado na grade não está protegido.

In [ ]:
for cond in CONDICOES:
    mostrar(FC.fig_percolacao(resumos, cond), f"18_percolacao_parametro_de_ordem_{cond}.png")
    mostrar(FC.fig_percolacao_mapas(resumos, cond), f"18_percolacao_mapas_{cond}.png")

## Figura 19: bloco central contra espalhados

- Diferença pareada: para cada cenário, $I_E(t) - I_C(t)$ calculado seed a seed (a seed $k$ tem a mesma população $S_0$, $T_0$ nas duas condições), com média e banda. Acima de zero, há mais infectados com a semente espalhada naquele instante.
- Métricas: fração da população infectada ao menos uma vez, pico máximo de infectados, passo do pico e fração de surtos falhos, lado a lado em C (círculo cheio) e E (quadrado vazio).

In [ ]:
for banda in BANDAS:
    for doenca in ("flu", "measles"):
        mostrar(FC.fig_diferenca_condicoes(resumos, doenca, banda), f"19_central_vs_espalhado_{doenca}_{banda}.png")
    mostrar(FC.fig_metricas_condicoes(resumos, banda), f"19_central_vs_espalhado_metricas_{banda}.png")

## Retoques sem editar o módulo

Tudo o que as funções devolvem é uma figura Matplotlib comum, então dá para alterar eixos, títulos e limites antes de `mostrar`. As constantes de cor dos módulos também podem ser trocadas, e valem para as figuras desenhadas depois da troca. Várias funções aceitam parâmetros de janela, por exemplo `FC.fig_regimes(resumos, "C", t_zoom=300)` ou `FC.fig_diferenca_condicoes(resumos, "flu", "ep", t_max_flu=1500)`. O exemplo abaixo está comentado; descomente e adapte.

In [ ]:
# FV.COR_GAUSSIANA = "#117733"                 # cor da alocação gaussiana nas figuras seguintes
# fam = FV.por_condicao(resumos, "C")
# fig = FV.fig_infectados(fam, "flu", "dp")
# fig.axes[0].set_ylim(0, 4000)                # os três painéis compartilham o eixo y
# fig.suptitle("Título alternativo", y=1.02)
# mostrar(fig, "05_flu_infectados_dp_C_retocada.png")

## Gravar tudo de uma vez

Equivale a rodar `python analyses/figuras_vacinacao.py`: grava todas as figuras e o `resumo_vacinacao.csv` em `SAIDA`, relendo os resultados. Ative a variável para executar.

In [ ]:
GRAVAR_TUDO = False

if GRAVAR_TUDO:
    FV.main(["--resultados", str(RESULTADOS), "--saida", str(SAIDA), "--dpi", str(DPI_SALVAR)])

## Transferir os resultados do cluster

Execute no terminal da máquina local (não neste notebook), na raiz do repositório, trocando `usuario@cluster` pelo endereço real. O filtro copia só os `.npz` reunidos, sem a pasta `results/pedacos/`:

```bash
mkdir -p results
rsync -avh --progress --include='*.npz' --exclude='*' usuario@cluster:~/small-world/results/ ./results/
```

Para trazer também os PNG e o CSV gerados no cluster:

```bash
rsync -avh usuario@cluster:~/small-world/analyses/plots/ ./analyses/plots/
```